## Job: Bronze to Silver

- 1. Verifica os nomes das colunas e padroniza para snake_case
- 2. Trata as colunas de descritores administrativos dos municípios
- 3. Trata os indicadores da pesquisa da planilha "Informática e Comunicação"
- 4. Trata os indicadores da pesquisa da planilha "Governança"
- 5. Verificação de integridade
    - unicidade da chave primária e correspondência da contagem de linhas (5.570 == 5.570)
- 6. Junção das tabelas de dados
- 7. Gravação na silver
- 8. Validação pós-gravação (Etapa 3)

In [0]:
%run ../environment/env_variables

In [0]:
%run ../environment/env_functions

In [0]:
import pyspark.sql.functions as F

In [0]:
# Caminhos onde as tabelas finais serão armazenadas
bronze_tbl_dict = f'{catalog_name}.{bronze_schema}.munic_raw_dict'
bronze_tbl_info = f'{catalog_name}.{bronze_schema}.munic_raw_info_data'
bronze_tbl_gov = f'{catalog_name}.{bronze_schema}.munic_raw_gov_data'

silver_tbl_dict = f'{catalog_name}.{silver_schema}.munic_clean_dict'
silver_tbl_data = f'{catalog_name}.{silver_schema}.munic_clean_data'

## Step 1 - Carga das tabelas bronze e verificação snake_case

In [0]:
# Carrega as tabelas bronze, filtradas pelo ano da edição da pesquisa
ano_edicao = int(reference_year) if isinstance(reference_year, (int, float, str)) else 2024

df_raw_dict = spark.table(bronze_tbl_dict).filter(F.col("ano_edicao") == ano_edicao)
df_raw_info = spark.table(bronze_tbl_info).filter(F.col("ano_edicao") == ano_edicao)
df_raw_gov = spark.table(bronze_tbl_gov).filter(F.col("ano_edicao") == ano_edicao)

In [0]:
# Verifica se todos os nomes de colunas estão padronizados em snake_case
import re

# Define uma expressão regular para validar snake_case
regex_expr = r'^[a-z0-9_]+$'

for name, df in [("munic_raw_dict", df_raw_dict), ("munic_raw_info", df_raw_info), ("munic_raw_gov", df_raw_gov)]:
    non_snake = [col for col in df.columns if not re.match(regex_expr, col)]

    print(f"{name}: {len(df.columns)} colunas, fora do snake_case: {non_snake or 'nenhuma'}")

## Step 2 - Tratamento dos descritores administrativos dos municípios

In [0]:
df_city_metainfo = df_raw_info.select("cod_munic", "desc_mun", "uf", "populacao", "faixa_populacao", "regiao")

In [0]:
df_city_metainfo.printSchema()

In [0]:
# 1. Transforma o cod_munic em string
# 2. Remove a coluna 'cod_munic'
# 3. Renomeia colunas para clareza
def treat_city_metainfo(df):
    return (
        df
        .withColumn("cod_ibge", F.col("cod_munic").cast("string"))
        .drop("cod_munic")
        .withColumnsRenamed({
            "desc_mun": "nome",
            "uf": "uf_sigla",
            "populacao": "populacao_total",
        })
    )


In [0]:
# Aplica o tratamento aos metadados dos municípios
df_city_metainfo = treat_city_metainfo(df_city_metainfo)
display(df_city_metainfo.limit(5))

## Step 3 - Tratamento dos indicadores da planilha "Informática e Comunicação"

In [0]:
# Seleciona apenas cod_munic e as colunas de indicadores MTIC da tabela info
# Descarta as colunas de descritores do município e metadados
mtic_cols = [c for c in df_raw_info.columns if c.startswith("mtic")]
df_info_indicators = df_raw_info.select("cod_munic", *mtic_cols)

print(f"Indicadores info: {len(mtic_cols)} colunas MTIC")
display(df_info_indicators.limit(5))

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# Trata as colunas de indicadores de um BLOCO da MUNIC (MTIC, MGOV, ...) LIDO DA
# BRONZE (DataFrame) — não do XLSX: a ingestão já aconteceu. Reutilizável entre
# blocos: só muda o prefixo das colunas ("mtic", "mgov").
# ─────────────────────────────────────────────────────────────────────────────

def treat_survey_indicators(df, column_prefix, dict_table_name, ano_edicao=None):
    """
    Trata as colunas de indicadores de um bloco da MUNIC (mtic/mgov), lido da
    bronze, mantendo todas como StringType (Ajuste 1 - Etapa 3).

    Cada coluna preserva os 6 estados originais da pesquisa ('Sim', 'Não', '-',
    'Não sabe informar', 'Não informou', 'Recusa'); apenas células vazias ou
    apenas com espaços tornam-se NULL. A classificação binária/categórica é
    mantida somente para fins de relatório.

    Parameters
    ----------
    df               : DataFrame lido da bronze (cod_munic + colunas de indicadores)
    column_prefix    : prefixo em minúsculas ("mtic" ou "mgov")
    dict_table_name  : nome totalmente qualificado da tabela de dicionário
    ano_edicao       : ano da edição, usado para filtrar o dicionário (opcional)

    Retorna um dict: df, binary_cols, categorical_cols, meanings
    """
    # Estados originais de não-resposta do IBGE — preservados nos dados (Ajuste 1);
    # usados apenas na classificação binária/categórica para relatório abaixo
    NON_RESPONSE = ["Recusa", "Não informou", "Não sabe informar", "Não sabe", "-"]

    # Carrega os significados do dicionário para este prefixo (códigos em maiúsculas no dict),
    # filtrando pela edição da pesquisa para não misturar significados entre anos
    df_dict = spark.table(dict_table_name)
    if ano_edicao is not None:
        df_dict = df_dict.filter(F.col("ano_edicao") == ano_edicao)
    meanings = {
        row["cod_variable"].lower(): row["meaning"]
        for row in df_dict.filter(
            F.col("cod_variable").rlike(f"^{column_prefix.upper()}")
        ).collect()
    }
    print(f"Carregados {len(meanings)} significados de {column_prefix.upper()} do dicionário")

    # Armazena a lista de colunas uma vez (evita RPCs de Analyze repetidos no Spark Connect)
    indicator_cols = [c for c in df.columns if c.startswith(column_prefix)]

    # collect_set em uma única passada para obter os valores distintos por coluna (1 job Spark)
    distinct_sets = df.agg(
        *[F.collect_set(c).alias(c) for c in indicator_cols]
    ).collect()[0].asDict()

    # Classificação: subconjunto de {"Sim", "Não"} → binária, caso contrário categórica
    binary_cols, categorical_cols = [], []
    for col_name in indicator_cols:
        meaningful = set(str(v) for v in distinct_sets[col_name]) - set(NON_RESPONSE) - {"None", "nan"}
        if meaningful <= {"Sim", "Não"}:
            binary_cols.append(col_name)
        else:
            categorical_cols.append(col_name)

    print(f"Binárias: {len(binary_cols)} | Categóricas: {len(categorical_cols)}")
    print(f"Categóricas: {categorical_cols}")

    # Ajuste 1 (Etapa 3): tratamento uniforme para toda coluna de indicador.
    # Mantém StringType, remove espaços das bordas e preserva os 6 estados originais;
    # NULL apenas para strings vazias ou compostas só de espaços.
    select_exprs = [F.col("cod_munic")]
    for col_name in indicator_cols:
        select_exprs.append(
            F.when(F.trim(F.col(col_name).cast("string")) == "", F.lit(None).cast("string"))
            .otherwise(F.trim(F.col(col_name).cast("string")))
            .alias(col_name)
        )

    treated_df = df.select(*select_exprs)

    return {
        "df": treated_df,
        "binary_cols": binary_cols,
        "categorical_cols": categorical_cols,
        "meanings": meanings,
    }

#### Aplicação do tratamento nos indicadores

In [0]:
# Aplica a função de tratamento reutilizável aos indicadores MTIC (Informática e Comunicação)
# column_prefix="mtic" casa com colunas como mtic011, mtic04, mtic12a1, etc.
# dict_table_name=bronze_tbl_dict é a tabela munic_raw_dict definida na célula 4
result_info = treat_survey_indicators(
    df_info_indicators,
    column_prefix="mtic",
    dict_table_name=bronze_tbl_dict,
    ano_edicao=ano_edicao
)

# Desempacota o dict de resultados em variáveis individuais
df_info_indicators = result_info["df"]                   # DataFrame tratado (todos os indicadores StringType, 6 estados preservados)
mtic_meanings = result_info["meanings"]                 # dict: {"mtic011": "Correio", ...}
binary_cols_info = result_info["binary_cols"]           # lista de 188 colunas binárias
categorical_cols_info = result_info["categorical_cols"]  # lista de 10 colunas categóricas

print(f"\nTratamento aplicado a {len(binary_cols_info) + len(categorical_cols_info)} colunas MTIC")


print("=== Schema após o tratamento (todas as colunas de indicadores MTIC devem ser string) ===")
df_info_indicators.printSchema()

# Check: nenhum indicador MTIC fora de StringType (Ajuste 1 — acumula no lote
# de check_igual, que fecha no resumo_validacoes do fail-fast pré-gravação)
non_string_mtic = [
    c for c in binary_cols_info + categorical_cols_info
    if df_info_indicators.schema[c].dataType.simpleString() != "string"
]
check_igual(len(non_string_mtic), 0, "tratamento MTIC: indicadores em StringType",
            contexto=f"fora de StringType: {non_string_mtic[:10]}")

## Step 4 - Tratamento dos indicadores da planilha "Governança"

In [0]:
# Seleciona apenas cod_munic e as colunas de indicadores MGOV da tabela gov
# Descarta as colunas de descritores do município e metadados
mgov_cols = [c for c in df_raw_gov.columns if c.startswith("mgov")]
df_gov_indicators = df_raw_gov.select("cod_munic", *mgov_cols)

print(f"Indicadores gov: {len(mgov_cols)} colunas MGOV")
display(df_gov_indicators.limit(5))

In [0]:
# Aplica a mesma função de tratamento reutilizável aos indicadores MGOV (Governança)
# column_prefix="mgov" casa com colunas como mgov01, mgov011a, mgov36, etc.
# Reutilizamos a mesma função — sem duplicação de código entre o tratamento MTIC e MGOV
result_gov = treat_survey_indicators(
    df_gov_indicators,
    column_prefix="mgov",
    dict_table_name=bronze_tbl_dict,
    ano_edicao=ano_edicao
)

# Desempacota os resultados
df_gov_indicators = result_gov["df"]                    # DataFrame tratado
mgov_meanings = result_gov["meanings"]                 # dict: {"mgov01": "...", ...}
binary_cols_gov = result_gov["binary_cols"]             # lista de colunas binárias
categorical_cols_gov = result_gov["categorical_cols"]  # lista de colunas categóricas

print(f"\nTratamento aplicado a {len(binary_cols_gov) + len(categorical_cols_gov)} colunas MGOV")

# Check: nenhum indicador MGOV fora de StringType (Ajuste 1 — acumula no lote
# de check_igual, que fecha no resumo_validacoes do fail-fast pré-gravação)
non_string_mgov = [
    c for c in binary_cols_gov + categorical_cols_gov
    if df_gov_indicators.schema[c].dataType.simpleString() != "string"
]
check_igual(len(non_string_mgov), 0, "tratamento MGOV: indicadores em StringType",
            contexto=f"fora de StringType: {non_string_mgov[:10]}")

display(df_gov_indicators.limit(5))

## Step 5 - Verificação de integridade
- unicidade da chave primária; contagem de linhas validada contra o **contrato da edição** no ledger `pipeline_manifest` (declarado pela bronze ao gravar — sem números hardcoded)

In [0]:
# Verifica a unicidade da chave primária nas duas tabelas de origem
info_count = df_raw_info.count()
gov_count = df_raw_gov.count()

# O .distinct() é OBRIGATÓRIO aqui: .select() sozinho não deduplica, então sem ele
# estas contagens apenas recontariam as linhas e as asserções de unicidade abaixo
# passariam sempre, independentemente de quantas chaves duplicadas a tabela tenha.
info_distinct = df_raw_info.select("cod_munic").distinct().count()
gov_distinct = df_raw_gov.select("cod_munic").distinct().count()

print(f"Tabela info: {info_count} linhas, {info_distinct} cod_munic distintos")
print(f"Tabela gov:  {gov_count} linhas, {gov_distinct} cod_munic distintos")

assert info_count == info_distinct, f"cod_munic duplicado encontrado na tabela info"
assert gov_count == gov_distinct, f"cod_munic duplicado encontrado na tabela gov"
assert info_count == gov_count, f"Contagens de linhas divergentes: {info_count} != {gov_count}"

# Contrato da edição no ledger (pipeline_manifest): a bronze declarou linhas /
# shape / recusa ao gravar — o fail-fast valida contra isso, em vez do 5570
# hardcoded (que só valia enquanto a bronze tinha uma única edição carregada).
manifest_bronze = contrato_da_edicao(etapa="bronze", ano_edicao=ano_edicao)
# Check: linhas da tabela info da bronze == linhas declaradas no ledger
check_igual(info_count, manifest_bronze["linhas"], "linhas_info_vs_contrato")

print(f"\nContrato da bronze ({ano_edicao}): linhas {manifest_bronze['linhas']}, "
      f"indicadores {manifest_bronze['indicadores']}, recusa {manifest_bronze['recusa_total']}")

## Step 6 - Junção das tabelas de dados

In [0]:
# Junta as tabelas de indicadores por cod_munic (inner join)
df_merged_indicators = df_info_indicators.join(df_gov_indicators, on="cod_munic", how="inner")

# Converte cod_munic para string como cod_ibge para a junção com os metadados dos municípios
df_merged_indicators = (
    df_merged_indicators
    .withColumn("cod_ibge", F.col("cod_munic").cast("string"))
    .drop("cod_munic")
)

# Junta com os metadados dos municípios por cod_ibge
df_silver_data = df_city_metainfo.join(df_merged_indicators, on="cod_ibge", how="inner")

merged_count = df_silver_data.count()

print(f"Linhas na tabela juntada: {merged_count}")
print(f"Colunas na tabela juntada: {len(df_silver_data.columns)}")

# Um inner join descarta silenciosamente linhas cuja chave falta em qualquer um dos lados.
# Assumimos via assert que a contagem de municípios sobreviveu às duas junções — contra o
# contrato da edição no ledger, não contra um número digitado.
assert merged_count == manifest_bronze["linhas"], (
    f"Linhas perdidas na junção: esperadas {manifest_bronze['linhas']}, obtidas {merged_count}"
)

## Step 7 - Gravação na silver

In [0]:
# O tratamento dos indicadores nos Steps 3 e 4 reconstrói os DataFrames com select(),
# o que descarta as colunas de linhagem gravadas pelo job da bronze. Reanexamos aqui
# para que as tabelas silver ainda consigam responder "de onde esta linha veio e quando".
#
# reference_year e munic_source_url vêm do env_variables (notebook de ambiente).

df_silver_data = df_silver_data.withColumns({
    "ano_edicao": F.lit(ano_edicao),
    "dataset_name": F.lit("IBGE_MUNIC"),
    "reference_year": F.lit(reference_year),
    "source_url": F.lit(munic_source_url),
    "source_file": F.lit(munic_file),
    "silver_processed_at": F.current_timestamp(),
})

# Posiciona a chave da entidade primeiro, depois a chave de partição: cod_ibge e então ano_edicao
all_cols = df_silver_data.columns
other_cols = [c for c in all_cols if c not in ("ano_edicao", "cod_ibge")]
df_silver_data = df_silver_data.select("cod_ibge", "ano_edicao", *other_cols)

display(
    df_silver_data.select(
        "cod_ibge", "ano_edicao", "dataset_name", "reference_year", "silver_processed_at"
    ).limit(5)
)

#### Fail-fast: validação contra o contrato do ledger

- Os checks `check_igual` (célula acima da migração de schema) rodam **antes de qualquer escrita na silver** — inclusive da migração boolean→StringType — então um run reprovado morre sem deixar a tabela em estado meio-migrado
- O que eram asserts hardcoded (360 indicadores / 5.570 linhas / 2.520 recusas) agora valida contra a linha `bronze/edição` do `pipeline_manifest`: adiciona uma edição nova e os números acompanham automaticamente
- A tabela gravada é reverificada no Step 8 (a checagem pré-gravação valida os dados, a pós-gravação valida o artefato)

*Detalhe: contraparte pré-gravação das checagens do "Ajuste 1", validadas novamente no Step 8.*

In [0]:
# Fail-fast contra o contrato da bronze no ledger (docs/arquitetura/
# pipeline_manifest.md, §5.2): os checks acumulam no lote (check_igual) e o
# resumo_validacoes interrompe o run com TODAS as falhas de uma vez — antes
# da primeira escrita na silver (migração de schema e MERGE abaixo).

# Silver por edição: descarta colunas 100% nulas (só existem em outra edição)
indicator_cols_todos = [c for c in df_silver_data.columns if c.startswith(("mtic", "mgov"))]
com_dados = set(colunas_com_dados(df_silver_data, indicator_cols_todos))   # env_functions
colunas_nulas = [c for c in indicator_cols_todos if c not in com_dados]

if colunas_nulas:
    print(f"Descartando {len(colunas_nulas)} coluna(s) 100% nulas nesta edição "
          f"(existem só em outra): {colunas_nulas[:8]}{'...' if len(colunas_nulas) > 8 else ''}")
    df_silver_data = df_silver_data.drop(*colunas_nulas)
indicator_cols_final = [c for c in df_silver_data.columns if c.startswith(("mtic", "mgov"))]

# Check: nº de municípios == linhas declaradas pela bronze
check_igual(df_silver_data.count(), manifest_bronze["linhas"], "linhas_municipios")
# Check: shape com dados da edição == indicadores declarados pela bronze
check_igual(len(indicator_cols_final), manifest_bronze["indicadores"], "indicadores_com_dados",
              contexto=f"detalhes da bronze: {manifest_bronze['detalhes']}")

non_string_final = [
    c for c in indicator_cols_final
    if df_silver_data.schema[c].dataType.simpleString() != "string"
]
# Check: nenhum indicador fora de StringType (Ajuste 1)
check_igual(len(non_string_final), 0, "indicadores_em_stringtype",
              contexto=f"fora de StringType: {non_string_final[:10]}")

# Check: total de células 'Recusa' == recusa declarada pela bronze (invariante)
recusa_final = celulas_recusa(df_silver_data, indicator_cols_final)   # env_functions
check_igual(recusa_final, manifest_bronze["recusa_total"], "recusa_preservada")

# Fecha o lote: qualquer FAIL interrompe o run AQUI (fail-fast), antes da migração e do MERGE
resumo_validacoes()

#### Migração: boolean → StringType

- A silver pré-Etapa-3 armazenava os indicadores binários como `boolean`, destruindo os 6 estados originais (`Sim`, `Não`, `-`, `Não sabe informar`, `Não informou`, `Recusa`)
- O MERGE não altera tipos de coluna, então ao detectar o schema legado a tabela é reconstruída in-place com `overwriteSchema=true`, registrando a refatoração no `userMetadata`
- O MERGE normal do Step 7 então regrava todas as linhas da edição corrente a partir das fontes bronze

*Detalhe: isto implementa a refatoração do "Ajuste 1" exigida pela especificação da Etapa 3 (`docs/arquitetura/gold_dimensional_model.md`, Seção 1.1).*

In [0]:
# Ajuste 1 (Etapa 3): a silver pré-refatoração armazenava indicadores binários como boolean, destruindo os 6 estados originais. O MERGE não altera tipos de coluna, então ao
# detectar o schema legado a tabela é reconstruída in-place com overwriteSchema e a
# refatoração é registrada no userMetadata. O MERGE abaixo então regrava todas as
# linhas da edição corrente a partir das fontes bronze.
if spark.catalog.tableExists(silver_tbl_data):
    legacy_target = spark.table(silver_tbl_data)
    non_string_indicators = [
        c for c in legacy_target.columns
        if c.startswith(("mtic", "mgov"))
        and legacy_target.schema[c].dataType.simpleString() != "string"
    ]
    if non_string_indicators:
        print(f"Schema legado detectado: convertendo {len(non_string_indicators)} colunas de indicadores para StringType")
        (
            legacy_target
            .withColumns({c: F.col(c).cast("string") for c in non_string_indicators})
            .write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .partitionBy("ano_edicao")
            .option("userMetadata", "Refatoracao Pos-Revisao Etapa 3: Preservacao dos 6 estados originais da MUNIC")
            .saveAsTable(silver_tbl_data)
        )
        print("Migração concluída - o MERGE abaixo regrava todas as linhas da edição corrente")
    else:
        print("Schema da silver já compatível (todos os indicadores são StringType)")
else:
    print(f"{silver_tbl_data} ainda não existe - migração desnecessária")

#### Gravação das tabelas na Silver

In [0]:
# Grava na silver usando MERGE (upsert), particionada por ano_edicao
# upsert_delta_table é importado do notebook env_functions

# Limpa o dicionário - posiciona cod_variable (chave da entidade) primeiro, depois ano_edicao (chave de partição)
df_clean_dict = df_raw_dict.select("cod_variable", "ano_edicao", "meaning")

# Tabela de dicionário - chave do MERGE: ano_edicao + cod_variable
upsert_delta_table(
    df_clean_dict,
    silver_tbl_dict,
    ["ano_edicao", "cod_variable"],
    ano_edicao,
    f"Dicionário MUNIC limpo - edição {ano_edicao}"
)

# Tabela de dados - chave do MERGE: ano_edicao + cod_ibge
upsert_delta_table(
    df_silver_data,
    silver_tbl_data,
    ["ano_edicao", "cod_ibge"],
    ano_edicao,
    f"Dados da pesquisa MUNIC limpos e juntados - edição {ano_edicao}"
)

print(f"\nLinhas do dict: {df_clean_dict.count()}")
print(f"Linhas de dados: {df_silver_data.count()}")

## Step 8 - Validação pós-gravação (Etapa 3)

- Todas as colunas de indicadores **com dados da edição** em `silver.munic_clean_data` devem ser `StringType` e bater com o shape declarado pela bronze no ledger
- O total de células 'Recusa' deve bater com o `recusa_total` do contrato da edição (invariante bronze→silver)
- Nenhum resíduo booleano (strings 'true'/'false') pode restar após a migração

*Detalhe: as checagens acima são a validação imediata pós-ajuste exigida pela refatoração do "Ajuste 1" na especificação (`docs/arquitetura/gold_dimensional_model.md`, Seção 1.1).*

In [0]:
from functools import reduce

silver_check = spark.table(silver_tbl_data).filter(F.col("ano_edicao") == ano_edicao)
# Silver POR EDIÇÃO: no schema união da tabela, colunas de outras edições chegam
# 100% nulas para a corrente — o contrato declara o shape COM DADOS da edição
# (360 em 2024, 307 em 2019), e a pós-gravação confere o artefato gravado.
todas_indicadoras_silver = [c for c in silver_check.columns if c.startswith(("mtic", "mgov"))]
indicator_cols_silver = colunas_com_dados(silver_check, todas_indicadoras_silver)   # env_functions


# Check: shape com dados no artefato gravado == indicadores da bronze
check_igual(len(indicator_cols_silver), manifest_bronze["indicadores"],
              "silver pós-gravação: indicadores com dados",
              contexto=f"detalhes da bronze: {manifest_bronze['detalhes']}")

non_string = [
    c for c in indicator_cols_silver
    if silver_check.schema[c].dataType.simpleString() != "string"
]

# Check: todos os indicadores gravados em StringType (Ajuste 1)
check_igual(len(non_string), 0, "silver pós-gravação: indicadores em StringType",
              contexto=f"fora de StringType: {non_string[:10]}")

# Recusa preservada: invariante da cadeia — o total declarado pela bronze chega intacto
recusa_total = celulas_recusa(silver_check, indicator_cols_silver)   # env_functions
# Check: recusa preservada no artefato gravado (invariante bronze→silver)
check_igual(recusa_total, manifest_bronze["recusa_total"], "silver pós-gravação: recusa preservada")

# Nenhum resíduo booleano deve restar após a migração + regravação do MERGE
boolean_residue = silver_check.agg(
    reduce(
        lambda a, b: a + b,
        [F.sum(F.when(F.col(c).isin("true", "false"), 1).otherwise(0)) for c in indicator_cols_silver],
    ).alias("residue")
).collect()[0]["residue"]

# Check: nenhum resíduo 'true'/'false' da migração boolean→string
check_igual(boolean_residue, 0, "silver pós-gravação: sem resíduo booleano ('true'/'false')",
              contexto=f"resíduo: {boolean_residue} células")

# Fecha o lote: qualquer FAIL interrompe o run AQUI (fail-fast), antes de declarar a silver no ledger
resumo_validacoes()

print(f"Ajuste 1 OK: {len(indicator_cols_silver)} indicadores StringType | células 'Recusa' = {recusa_total} | resíduo booleano = 0")

In [0]:
# Declara o contrato da SILVER no ledger: o que a etapa produziu para esta edição.
# O silver_to_gold valida contra isso no Step 1 (fronteira silver→gold) — o shape
# da edição (360/307), a recusa preservada e o mesmo SHA-256 da origem.
gravar_linha_manifest(
    etapa="silver",
    ano_edicao=ano_edicao,
    tabela_destino=f"{silver_tbl_data};{silver_tbl_dict}",
    sha256_xlsx=manifest_bronze["sha256_xlsx"],   # mesmo insumo da bronze (elo de linhagem)
    linhas=df_silver_data.count(),
    indicadores=len(indicator_cols_final),
    recusa_total=recusa_final,
    detalhes={
        "schema_total": str(len(silver_check.columns)),
        "estados_preservados": "6",
        "tipo_resposta": "StringType",
        "dict_linhas": str(df_clean_dict.count()),
    },
)